# YOLOv8 segmentation experiment for Sentinel 1 oil spills

This notebook trains and evaluates YOLOv8n-seg on the same deterministic 1,020 scene training split and 180 scene validation split used by the U-Net baseline. It produces native YOLO segmentation metrics plus common semantic-union IoU, Dice, precision, and recall for the final comparison report.

Before running: choose **Runtime > Change runtime type > GPU**, then upload `oil_spill_yolov8_part1.zip` to `My Drive/oil_spill_colab/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/oil_spill_colab')
ZIP_PATH = DRIVE_DIR / 'oil_spill_yolov8_part1.zip'
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
assert ZIP_PATH.is_file(), f'Upload the dataset archive first: {ZIP_PATH}'
print('Dataset archive:', ZIP_PATH, ZIP_PATH.stat().st_size, 'bytes')

In [ ]:
%pip install -q ultralytics==8.4.140

In [ ]:
import platform, torch, ultralytics
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('Ultralytics:', ultralytics.__version__)
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU is not active. Change the Colab runtime to GPU and reconnect.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import shutil

EXTRACT_PARENT = Path('/content/datasets')
DATA_ROOT = EXTRACT_PARENT / 'yolov8_part1'
if not (DATA_ROOT / 'train' / 'images').is_dir():
    EXTRACT_PARENT.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(ZIP_PATH), str(EXTRACT_PARENT))

train_images = sorted((DATA_ROOT / 'train' / 'images').glob('*.tif'))
val_images = sorted((DATA_ROOT / 'val' / 'images').glob('*.tif'))
assert len(train_images) == 2040, len(train_images)
assert len(val_images) == 360, len(val_images)

DATA_YAML = Path('/content/oil_spill_colab.yaml')
DATA_YAML.write_text(
    f"path: {DATA_ROOT.as_posix()}\ntrain: train/images\nval: val/images\nnames:\n  0: oil_spill\n",
    encoding='utf-8',
)
print('Ready:', len(train_images), 'train tiles and', len(val_images), 'validation tiles')

## Train YOLOv8n segmentation

The run uses 256 pixel tiles, the fixed seed, pretrained YOLOv8n-seg weights, mixed precision, and early stopping within a 20 epoch budget. Results are written directly to Google Drive so they survive a runtime disconnect.

In [ ]:
from ultralytics import YOLO

RUNS_DIR = DRIVE_DIR / 'runs'
RUN_NAME = 'yolov8n_seg_part1_e20'
model = YOLO('yolov8n-seg.pt')
train_results = model.train(
    task='segment',
    data=str(DATA_YAML),
    epochs=20,
    patience=5,
    imgsz=256,
    batch=32,
    device=0,
    workers=2,
    seed=42,
    deterministic=True,
    amp=True,
    cache=False,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
)
RUN_DIR = Path(train_results.save_dir)
BEST_WEIGHTS = RUN_DIR / 'weights' / 'best.pt'
assert BEST_WEIGHTS.is_file(), BEST_WEIGHTS
print('Best checkpoint:', BEST_WEIGHTS)

## Native YOLO validation metrics

In [ ]:
best_model = YOLO(str(BEST_WEIGHTS))
native = best_model.val(
    data=str(DATA_YAML),
    split='val',
    imgsz=256,
    batch=32,
    device=0,
    workers=2,
    project=str(RUNS_DIR),
    name=RUN_NAME + '_validation',
    exist_ok=True,
)
print(native.results_dict)
print('Speed milliseconds per image:', native.speed)

## Common semantic union metrics

YOLO instance masks are merged into one oil spill mask per tile. Confidence is swept on the fixed validation set so the output can be compared with the U-Net binary segmentation metrics.

In [ ]:
import cv2
import json
import numpy as np
from datetime import datetime, timezone

thresholds = [0.05, 0.10, 0.20, 0.25, 0.30, 0.50, 0.70]
counts = {str(t): {'tp': 0, 'fp': 0, 'fn': 0} for t in thresholds}

def label_mask(label_path, height, width):
    mask = np.zeros((height, width), dtype=np.uint8)
    if not label_path.is_file():
        return mask
    for line in label_path.read_text(encoding='utf-8').splitlines():
        values = [float(x) for x in line.split()]
        if len(values) < 7:
            continue
        xy = np.asarray(values[1:], dtype=np.float32).reshape(-1, 2)
        xy[:, 0] *= width
        xy[:, 1] *= height
        polygon = np.rint(xy).astype(np.int32)
        cv2.fillPoly(mask, [polygon], 1)
    return mask.astype(bool)

prediction_stream = best_model.predict(
    source=[str(p) for p in val_images],
    imgsz=256,
    batch=32,
    device=0,
    conf=min(thresholds),
    retina_masks=True,
    stream=True,
    verbose=False,
)

for result in prediction_stream:
    height, width = result.orig_shape
    image_path = Path(result.path)
    gt = label_mask(DATA_ROOT / 'val' / 'labels' / f'{image_path.stem}.txt', height, width)
    if result.masks is None:
        masks = np.zeros((0, height, width), dtype=bool)
        confidences = np.zeros((0,), dtype=np.float32)
    else:
        masks = result.masks.data.detach().cpu().numpy() > 0.5
        confidences = result.boxes.conf.detach().cpu().numpy()
        if masks.shape[-2:] != (height, width):
            masks = np.stack([cv2.resize(m.astype(np.uint8), (width, height), interpolation=cv2.INTER_NEAREST) > 0 for m in masks])
    for threshold in thresholds:
        selected = masks[confidences >= threshold]
        pred = np.any(selected, axis=0) if len(selected) else np.zeros_like(gt)
        key = str(threshold)
        counts[key]['tp'] += int(np.logical_and(pred, gt).sum())
        counts[key]['fp'] += int(np.logical_and(pred, ~gt).sum())
        counts[key]['fn'] += int(np.logical_and(~pred, gt).sum())

def metric_row(c):
    tp, fp, fn = c['tp'], c['fp'], c['fn']
    return {
        **c,
        'iou': tp / max(tp + fp + fn, 1),
        'dice': (2 * tp) / max(2 * tp + fp + fn, 1),
        'precision': tp / max(tp + fp, 1),
        'recall': tp / max(tp + fn, 1),
    }

semantic_by_confidence = {k: metric_row(v) for k, v in counts.items()}
best_confidence, best_semantic = max(semantic_by_confidence.items(), key=lambda item: item[1]['iou'])
print('Best semantic confidence:', best_confidence)
print(best_semantic)

In [ ]:
def jsonable(value):
    if isinstance(value, dict):
        return {str(k): jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [jsonable(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    return value

comparison = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'dataset': {
        'train_scenes': 1020, 'validation_scenes': 180,
        'train_tiles': 2040, 'validation_tiles': 360,
        'image_size': 256, 'channels': ['VV', 'VH', 'scaled_VV_minus_VH'],
    },
    'yolov8': {
        'ultralytics_version': ultralytics.__version__,
        'torch_version': torch.__version__,
        'gpu': torch.cuda.get_device_name(0),
        'model': 'yolov8n-seg.pt', 'epochs_budget': 20, 'seed': 42,
        'best_weights': str(BEST_WEIGHTS),
        'native_validation': jsonable(native.results_dict),
        'speed_ms_per_image': jsonable(native.speed),
        'semantic_by_confidence': semantic_by_confidence,
        'best_semantic_confidence': float(best_confidence),
        'best_semantic_metrics': best_semantic,
    },
    'unet_reference': {
        'checkpoint': 'unet_part1_epoch1.pt', 'epochs': 1,
        'best_threshold': 0.70, 'iou': 0.308307,
        'dice': 0.47131211562712966,
        'precision': 0.37687570162900563,
        'recall': 0.6289000576129438,
    },
}
METRICS_PATH = DRIVE_DIR / 'yolov8_colab_comparison_metrics.json'
METRICS_PATH.write_text(json.dumps(comparison, indent=2), encoding='utf-8')
RESULT_ARCHIVE = shutil.make_archive(str(DRIVE_DIR / 'yolov8_colab_results'), 'zip', root_dir=RUN_DIR)
print('Metrics:', METRICS_PATH)
print('Run archive:', RESULT_ARCHIVE)

## Finish

Download `yolov8_colab_comparison_metrics.json` or copy it into this Codex project. The final comparison report will use these measured values and the saved training artifacts.

In [ ]:
from google.colab import files
files.download(str(METRICS_PATH))